# 실습7 · 동의어와 불용어 — PostgreSQL 전문검색의 사전 체인

형태소 분석 · 동의어 · 불용어는 pgvector 가 아니라 **PostgreSQL 전문검색(tsearch)** 의 일이다.
한국어 형태소 분석은 textsearch_ko(mecab) 가 맡고, 동의어 · 불용어는 그 앞에 **사전(dictionary)** 을 끼워 넣어 붙인다.
강의 : 「키워드 검색의 문제」 · 「PG ts_rank」 · 「BM25」

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 연결과 평가 도구 |
| 3 | 파서가 이미 버리는 것 — 조사 · 어미는 품사로 |
| 4 | 불용어 후보 — 코퍼스에서 가장 흔한 형태소 |
| 5 | 질의에서 불용어 빼기 — ts_rank 와 BM25 평가 |
| 6 | 동의어 ① 질의 다시 쓰기(ts_rewrite) — 파일 · 재색인 없이 |
| 7 | 동의어 · 불용어 ② 사전 파일로 설정 만들기 |
| 8 | 동의어 ③ dict_xsyn — 질의 쪽에 쓰면 AND 가 된다 |
| 9 | 문서에 적용해 보고 되돌리기 — tsv 재계산 · BM25 트리거 |
| 10 | 정리 |

- 사전 파일(`db/tsearch/korean_lab.*`)이 DB 이미지에 들어 있어야 한다. 없으면 4-1 에서 `docker compose up -d --build db`
- 셀 9 는 docs 전체의 tsv 를 바꿨다가 **되돌린다**. 그동안 다른 노트북의 검색은 기다린다

**Run → Run All Cells** (약 3분 — 셀 5 약 40초 · 셀 8 약 20초 · 셀 9 약 1분 30초)

### 셀 1 · 설정
- `STOP_N` : 셀 5 에서 질의에서 뺄 불용어 수(코퍼스 문서 빈도 상위 N 개)
- `SYNONYMS` : 셀 6 의 질의 다시 쓰기 규칙 (이 단어가 오면 → 이 OR 질의로)

In [ ]:
# 셀 1 · 설정
DB_URL = "postgresql://lab:lab@db:5432/lab"
EVAL_N = 500                       # 평가 질문 수 (실습3·4·5 와 같은 질문)
STOP_N = [0, 10, 30, 100]          # 셀 5 : 질의에서 뺄 불용어 수
SYNONYMS = [                       # 셀 6 : 단어 → 바꿀 질의 (한 방향으로만)
    ("스마트폰", "스마트폰 | 휴대폰 | 핸드폰"),
    ("인공지능", "인공지능 | ai"),
]

### 셀 2 · 연결과 평가 도구
- `freq` : 코퍼스 형태소별 문서 수(`ts_stat`) · `stop_q` : 질의에서 뺄 형태소 · `set_stop(n)` : 상위 n 개로 채움
- `ts_rank_sql(설정)` · `bm25_sql(설정)` : 질문 500개를 그 설정으로 형태소 분석하고 `stop_q` 를 뺀 뒤 상위 10

In [ ]:
# 셀 2 · 연결과 평가 도구
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg
from pgvector.psycopg import register_vector

plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False, "figure.dpi": 110})
pd.set_option("display.max_colwidth", 70)

conn = psycopg.connect(DB_URL, autocommit=True)
conn.prepare_threshold = None      # SET 으로 바꾼 설정이 매번 반영되게 (실행 계획 재사용 끔)
register_vector(conn)              # numpy 배열 ↔ vector 타입


def sql(query, params=None):
    """SELECT 결과를 표(DataFrame)로"""
    cur = conn.execute(query, params)
    return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description]) if cur.description else None


# 평가용 질문 : 11개마다 하나씩 고른 EVAL_N 개. 모든 노트북이 같은 질문으로 잰다
E = f"(SELECT id, doc_id AS gold, question, embedding, emb128 FROM questions WHERE id % 11 = 0 ORDER BY id LIMIT {EVAL_N}) e"
RESULTS = Path("results/검색평가.csv")


def evaluate(name, query, params=None):
    """query 는 (질문 id, 정답 지문 id, 상위 문서 id 배열) 을 질문마다 한 행씩 돌려주는 SQL.
    정답 지문이 상위 10위 안이면 Hit, MRR = 정답 순위 역수의 평균. 결과는 RESULTS 에 누적"""
    print(f"{name} …", end=" ", flush=True)          # 방법 하나가 끝날 때마다 한 줄씩 보이게
    t = time.perf_counter()
    rows = conn.execute(query, params).fetchall()
    sec = time.perf_counter() - t
    rank = [list(ids[:10]).index(g) + 1 if g in ids[:10] else None for _, g, ids in rows]
    res = {"방법": name, "Hit@10": round(np.mean([r is not None for r in rank]), 3),
           "MRR": round(np.mean([1 / r if r else 0 for r in rank]), 3), "질문수": len(rows), "초": round(sec, 1)}
    RESULTS.parent.mkdir(exist_ok=True)
    new = pd.DataFrame([res])
    if RESULTS.exists():
        old = pd.read_csv(RESULTS)
        new = pd.concat([old[old["방법"] != name], new])
    new.to_csv(RESULTS, index=False)
    print(f"Hit@10 {res['Hit@10']:.3f} · MRR {res['MRR']:.3f} · {res['초']}s", flush=True)
    return res


SHARE = conn.execute("SELECT setting FROM pg_config WHERE name = 'SHAREDIR'").fetchone()[0] + "/tsearch_data/"
assert conn.execute("SELECT (pg_stat_file(%s, true)).size IS NOT NULL",(SHARE + "korean_lab.syn",)).fetchone()[0], \
    "사전 파일이 DB 이미지에 없다 → 4-1 에서 docker compose up -d --build db 후 셀 1 부터 다시"

conn.execute("DROP TABLE IF EXISTS freq, stop_q")
conn.execute("CREATE TEMP TABLE freq AS SELECT word, ndoc FROM ts_stat('SELECT tsv FROM docs')")
conn.execute("CREATE TEMP TABLE stop_q (word text PRIMARY KEY)")


def set_stop(n):
    """질의에서 뺄 불용어 = 코퍼스 문서 빈도 상위 n 개"""
    conn.execute("TRUNCATE stop_q")
    conn.execute("INSERT INTO stop_q SELECT word FROM freq ORDER BY ndoc DESC, word LIMIT %s", (n,))


# 질의문 → 형태소 → stop_q 를 뺀 OR tsquery
ORQ_STOP = """(SELECT coalesce(string_agg(quote_literal(l), ' | '), '')::tsquery
               FROM unnest(tsvector_to_array(to_tsvector('{cfg}', {q}))) l WHERE l NOT IN (SELECT word FROM stop_q))"""

# bm25_search() 와 같은 식 (실습4 셀 2). 질의어를 {cfg} 로 나누고 stop_q 를 뺀다
BM25_SQL = """
  WITH s  AS (SELECT count(*)::float8 AS n, avg(doclen)::float8 AS avgdl FROM docs),
       qt AS (SELECT DISTINCT lexeme AS term FROM unnest(to_tsvector('{cfg}', {q}))
              WHERE lexeme NOT IN (SELECT word FROM stop_q)),
       w  AS (SELECT qt.term, ln(1 + (s.n - d.df + 0.5) / (d.df + 0.5)) AS idf
              FROM qt JOIN bm25_df d ON d.term = qt.term CROSS JOIN s)
  SELECT t.doc_id FROM w
  JOIN bm25_tf t ON t.term = w.term JOIN docs d ON d.id = t.doc_id CROSS JOIN s
  GROUP BY t.doc_id
  ORDER BY sum(w.idf * t.tf * 2.2 / (t.tf + 1.2 * (0.25 + 0.75 * d.doclen / s.avgdl))) DESC, t.doc_id
  LIMIT 10"""


def ts_rank_sql(cfg="korean"):
    return f"""SELECT e.id, e.gold, ARRAY(SELECT d.id FROM docs d WHERE d.tsv @@ x.tq
                                           ORDER BY ts_rank(d.tsv, x.tq) DESC, d.id LIMIT 10)
               FROM {E} CROSS JOIN LATERAL (SELECT {ORQ_STOP.format(cfg=cfg, q='e.question')} AS tq) x"""


def bm25_sql(cfg="korean"):
    return f"SELECT e.id, e.gold, ARRAY({BM25_SQL.format(cfg=cfg, q='e.question')}) FROM {E}"


N = conn.execute("SELECT count(*) FROM docs").fetchone()[0]
print("지문", N, "개 · 형태소 종류", conn.execute("SELECT count(*) FROM freq").fetchone()[0], "개 · 사전 파일 위치", SHARE)

### 셀 3 · 파서가 이미 버리는 것
`ts_debug` 는 문장을 조각(token)으로 나누고 각 조각을 어느 사전이 무엇으로 바꿨는지 보여 준다.
- textsearch_ko 의 파서는 mecab 품사로 **조사 · 어미 · 접속부사를 `blank` 로 내보낸다** → 사전에 가지도 않고 버려진다
- 한국어의 "불용어 처리"는 목록이 아니라 품사로 이미 되어 있다. 목록 사전은 그 위에 더하는 것

In [ ]:
# 셀 3 · 파서가 이미 버리는 것
S3 = "김철수 기자가 인공지능 반도체를 만들었습니다. 그리고 AI 칩은 빠르다"
D3 = sql("""SELECT alias AS 종류, token AS 조각, array_to_string(dictionaries, ',') AS 사전,
                   array_to_string(lexemes, ',') AS 색인형태소
            FROM ts_debug('korean', %s)""", (S3,))
D3 = D3[D3["조각"].str.strip() != ""].copy()
D3["결과"] = np.where(D3["종류"] == "blank", "버림 (품사)", "색인")
display(D3.set_index("조각").T)
sql("SELECT to_tsvector('korean', %s) AS tsv", (S3,))

### 셀 4 · 불용어 후보 — 코퍼스에서 가장 흔한 형태소
`ts_stat` 으로 형태소마다 몇 개 지문에 나오는지 센다. 상위 형태소는 거의 모든 지문에 있어 검색어로서 구별력이 없다.
- BM25 는 IDF 로 이런 형태소를 이미 0 가까이 누른다(오른쪽 열). ts_rank 에는 IDF 가 없다

In [ ]:
# 셀 4 · 불용어 후보
F4 = sql("""SELECT word AS 형태소, ndoc AS 문서수, round(100.0 * ndoc / %(n)s, 1) AS "문서비율%%",
                   round(ln(1 + (%(n)s - ndoc + 0.5) / (ndoc + 0.5))::numeric, 3) AS idf
            FROM freq ORDER BY ndoc DESC, word LIMIT 30""", {"n": N})
fig, ax = plt.subplots(figsize=(13, 3.4))
ax.bar(F4["형태소"], F4["문서비율%"], color="#3D484A")
ax.set_ylabel("문서 비율 %"); ax.set_title(f"문서 빈도 상위 30 형태소 (지문 {N}개 중)")
plt.show()
F4.set_index("형태소").T

### 셀 5 · 질의에서 불용어 빼기 — ts_rank 와 BM25
질문 500개를 형태소 분석한 뒤 셀 4 의 상위 N 개를 빼고 검색한다(0 = 실습3 · 4 와 같은 조건). 약 40초.
- 불용어 목록은 질문이 아니라 **코퍼스 빈도**로 정했다 → 채점 문항에 맞춘 값이 아니다

In [ ]:
# 셀 5 · 질의에서 불용어 빼기
rows5 = []
for n in STOP_N:
    set_stop(n)
    rows5.append({**evaluate(f"07 OR ts_rank 질의 불용어 {n}", ts_rank_sql()), "불용어 수": n, "방법": "ts_rank (OR)"})
    rows5.append({**evaluate(f"07 BM25 질의 불용어 {n}", bm25_sql()), "불용어 수": n, "방법": "BM25"})
set_stop(0)
R5 = pd.DataFrame(rows5)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for a, m in zip(axes, ["Hit@10", "MRR"]):
    for meth, c in [("ts_rank (OR)", "#FF7043"), ("BM25", "#3D484A")]:
        s = R5[R5["방법"] == meth]
        a.plot(s["불용어 수"], s[m], "-o", color=c, label=meth)
        for x, y in zip(s["불용어 수"], s[m]):
            a.annotate(f"{y:.3f}", (x, y), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8)
    a.margins(y=0.15); a.set_xlabel("질의에서 뺀 불용어 수 (코퍼스 문서 빈도 상위)"); a.set_title(m); a.legend(loc="center right")
plt.tight_layout(); plt.show()
R5.pivot(index="방법", columns="불용어 수", values=["Hit@10", "MRR"])

### 셀 6 · 동의어 ① 질의 다시 쓰기 — `ts_rewrite`
규칙 표(`aliases`)의 tsquery 를 찾아 다른 tsquery 로 바꾼다. **파일도 재색인도 필요 없다.** 질의 쪽에서만 동작.
- 규칙은 질의에 올 수 있는 표현마다 있어야 한다 — "휴대폰" 으로 물으면 셀 1 규칙으로는 안 펼쳐진다
- 함정 : 양방향 규칙을 함께 넣으면 바꾼 결과를 또 바꿔 같은 단어가 겹친다(마지막 표)

In [ ]:
# 셀 6 · 동의어 ① 질의 다시 쓰기
conn.execute("DROP TABLE IF EXISTS aliases")
conn.execute("CREATE TEMP TABLE aliases (t tsquery PRIMARY KEY, s tsquery)")
for word, rewrite in SYNONYMS:
    conn.execute("INSERT INTO aliases VALUES (to_tsquery('korean', %s), to_tsquery('korean', %s))", (word, rewrite))

Q6 = ["스마트폰 판매", "인공지능 반도체", "휴대폰 요금"]
REWRITE = "SELECT t, s FROM aliases"
display(sql(f"""SELECT q AS 질문, plainto_tsquery('korean', q)::text AS 원래질의,
                       ts_rewrite(plainto_tsquery('korean', q), '{REWRITE}')::text AS 바꾼질의,
                       (SELECT count(*) FROM docs WHERE tsv @@ plainto_tsquery('korean', q)) AS 원래_문서수,
                       (SELECT count(*) FROM docs WHERE tsv @@ ts_rewrite(plainto_tsquery('korean', q), '{REWRITE}')) AS 바꾼_문서수
                FROM unnest(%s::text[]) q""", (Q6,)))

# 함정 : 반대 방향 규칙(ai → ai | 인공지능)을 더하면
conn.execute("INSERT INTO aliases VALUES (to_tsquery('korean', 'ai'), to_tsquery('korean', 'ai | 인공지능'))")
sql(f"SELECT ts_rewrite(to_tsquery('korean', '인공지능'), '{REWRITE}')::text AS 양방향_규칙을_넣은_뒤")

### 셀 7 · 동의어 · 불용어 ② 사전 파일로 설정 만들기
사전 파일은 DB 서버의 `tsearch_data` 폴더에 있어야 한다(4-1 `db/tsearch/` → DB 이미지에 복사). 새 설정 `korean_lab` 은 기본 `korean` 을 복사하고, 한국어 조각(word)을 세 사전에 차례로 통과시킨다.

| 순서 | 사전 | 파일 | 하는 일 |
|---|---|---|---|
| 1 | `korean_lab_stop` (simple) | `korean_lab.stop` | 목록에 있으면 버림, 아니면 다음 사전으로(ACCEPT = false) |
| 2 | `korean_lab_syn` (synonym) | `korean_lab.syn` | 목록에 있으면 대표어로 바꿈, 아니면 다음 사전으로 |
| 3 | `korean_stem` (mecab) | – | 원래 한국어 사전 |

- 사전은 **mecab 이 자른 형태소**를 받는다. `전기자동차 → 전기차` 규칙은 파일에 있어도 안 먹는다(전기 + 자동차로 먼저 잘림). 규칙을 쓰기 전에 `ts_debug` 로 확인

In [ ]:
# 셀 7 · 사전 파일로 설정 만들기
for f in ["korean_lab.stop", "korean_lab.syn"]:
    print(f"── {f}\n" + conn.execute("SELECT pg_read_file(%s)", (SHARE + f,)).fetchone()[0].strip().replace("\n", " · "))

conn.execute("DROP TEXT SEARCH CONFIGURATION IF EXISTS korean_lab")
conn.execute("DROP TEXT SEARCH DICTIONARY IF EXISTS korean_lab_stop")
conn.execute("DROP TEXT SEARCH DICTIONARY IF EXISTS korean_lab_syn")
conn.execute("CREATE TEXT SEARCH DICTIONARY korean_lab_stop (TEMPLATE = simple, STOPWORDS = korean_lab, ACCEPT = false)")
conn.execute("CREATE TEXT SEARCH DICTIONARY korean_lab_syn (TEMPLATE = synonym, SYNONYMS = korean_lab)")
conn.execute("CREATE TEXT SEARCH CONFIGURATION korean_lab (COPY = korean)")
conn.execute("""ALTER TEXT SEARCH CONFIGURATION korean_lab
                ALTER MAPPING FOR word, hword, hword_part WITH korean_lab_stop, korean_lab_syn, korean_stem""")

S7 = "김철수 기자가 휴대폰 등 인공지능 기기와 전기자동차를 소개했다"
display(sql("""SELECT 'korean' AS 설정, to_tsvector('korean', %(s)s)::text AS tsv
               UNION ALL SELECT 'korean_lab', to_tsvector('korean_lab', %(s)s)::text""", {"s": S7}))
sql("""SELECT token AS 조각, dictionary AS 처리한_사전, coalesce(array_to_string(lexemes, ','), '') AS 결과
       FROM ts_debug('korean_lab', %s) WHERE alias <> 'blank'""", (S7,)).set_index("조각").T

### 셀 8 · 동의어 ③ dict_xsyn — 한 단어를 여러 동의어로 펼치기
`korean_lab.rules` 의 한 줄 = "이 단어 → 원래 단어 + 동의어들". 한 조각이 여러 형태소로 **같은 위치**에 나온다.
- 질의 쪽에 쓰면 : 같은 위치의 형태소들이 **AND** 로 묶여 오히려 좁아진다(PostgreSQL 동작, 영어 설정도 같다)
- 문서 쪽에 쓰면 : 지문 tsv 에 동의어가 함께 들어가 어느 표현으로 물어도 걸린다 → xsyn 은 **색인할 때** 쓰는 사전

In [ ]:
# 셀 8 · dict_xsyn
conn.execute("CREATE EXTENSION IF NOT EXISTS dict_xsyn")
conn.execute("DROP TEXT SEARCH CONFIGURATION IF EXISTS korean_xsyn")
conn.execute("DROP TEXT SEARCH DICTIONARY IF EXISTS korean_lab_xsyn")
conn.execute("CREATE TEXT SEARCH DICTIONARY korean_lab_xsyn (TEMPLATE = xsyn_template, RULES = korean_lab, KEEPORIG = true)")
conn.execute("CREATE TEXT SEARCH CONFIGURATION korean_xsyn (COPY = korean)")
conn.execute("ALTER TEXT SEARCH CONFIGURATION korean_xsyn ALTER MAPPING FOR word, hword, hword_part WITH korean_lab_xsyn, korean_stem")
conn.execute("ALTER TEXT SEARCH CONFIGURATION korean_xsyn ALTER MAPPING FOR asciiword, asciihword, hword_asciipart WITH korean_lab_xsyn, english_stem")

print("rules :", conn.execute("SELECT pg_read_file(%s)", (SHARE + "korean_lab.rules",)).fetchone()[0].strip().replace("\n", " · "))
display(sql("""SELECT '질의 쪽' AS 적용, plainto_tsquery('korean_xsyn', '스마트폰')::text AS 결과
               UNION ALL SELECT '문서 쪽', to_tsvector('korean_xsyn', '휴대폰 요금 인하')::text"""))

# 문서 쪽에 쓰면 : 지문을 korean_xsyn 으로 다시 분석해 '스마트폰' · 'ai' 로 물어 본다 (지문 전체 분석 약 20초)
sql("""WITH x AS MATERIALIZED (SELECT id, tsv AS 기본, to_tsvector('korean_xsyn', coalesce(title, '') || ' ' || content) AS 펼침 FROM docs)
       SELECT q AS 질의,
              count(*) FILTER (WHERE 기본 @@ plainto_tsquery('korean', q)) AS "기본 tsv",
              count(*) FILTER (WHERE 펼침 @@ plainto_tsquery('korean', q)) AS "문서 쪽 xsyn",
              count(*) FILTER (WHERE 기본 @@ plainto_tsquery('korean_xsyn', q)) AS "질의 쪽 xsyn"
       FROM x, unnest(ARRAY['스마트폰', 'ai']) q GROUP BY q""")

### 셀 9 · 문서에 적용해 보고 되돌리기
설정을 바꿔도 **저장된 `docs.tsv` 는 그대로**다. 새 설정을 쓰려면 tsv · doclen 을 다시 계산해 UPDATE 해야 하고, 그 UPDATE 가 실습1 의 **BM25 트리거**를 불러 역색인(bm25_tf · bm25_df)도 고친다.
- 한 트랜잭션 안에서 : 전체 지문을 `korean_lab` 으로 다시 분석 → 역색인 변화 확인 → 질문도 `korean_lab` 으로 평가 → **ROLLBACK**
- 끝나면 원래대로 돌아왔는지 확인하고, 버려진 행 정리(VACUUM). 약 1분 30초

In [ ]:
# 셀 9 · 문서에 적용해 보고 되돌리기
COUNT = "SELECT (SELECT count(*) FROM bm25_tf), (SELECT count(*) FROM bm25_df), (SELECT sum(doclen) FROM docs)"
state = {"적용 전": conn.execute(COUNT).fetchone()}
set_stop(0)
rows9 = []
with conn.transaction():
    t = time.perf_counter()
    conn.execute("""WITH src AS MATERIALIZED (
                      SELECT id, setweight(to_tsvector('korean_lab', coalesce(title, '')), 'A') ||
                                 setweight(to_tsvector('korean_lab', content), 'D') AS tsv FROM docs)
                    UPDATE docs d SET tsv = s.tsv, doclen = tsv_len(s.tsv) FROM src s WHERE d.id = s.id""")
    print(f"지문 {N}개 tsv 재계산 + BM25 트리거 : {time.perf_counter() - t:.1f}s")
    state["korean_lab 적용"] = conn.execute(COUNT).fetchone()
    rows9.append({**evaluate("07 OR ts_rank korean_lab 문서·질의", ts_rank_sql("korean_lab")), "설정": "korean_lab", "방법": "ts_rank (OR)"})
    rows9.append({**evaluate("07 BM25 korean_lab 문서·질의", bm25_sql("korean_lab")), "설정": "korean_lab", "방법": "BM25"})
    raise psycopg.Rollback()          # 트랜잭션 전체를 되돌린다 (예외는 여기서 끝난다)
state["ROLLBACK 후"] = conn.execute(COUNT).fetchone()
conn.execute("VACUUM (ANALYZE) docs, bm25_tf, bm25_df")
display(pd.DataFrame(state, index=["bm25_tf 행", "bm25_df 행 (형태소 종류)", "전체 형태소 수"]).T)

base = R5[R5["불용어 수"] == 0].assign(설정="korean (기본)")
R9 = pd.concat([base, pd.DataFrame(rows9)])[["설정", "방법", "Hit@10", "MRR"]]
R9.pivot(index="방법", columns="설정", values=["Hit@10", "MRR"])

### 셀 10 · 정리

| 방법 | 파일 | 재색인 | 적용 범위 | 주의 |
|---|---|---|---|---|
| 품사로 버림 (파서) | – | – | 문서 · 질의 | 이미 기본 동작 |
| 질의 불용어 빼기 | – | – | 질의 | 목록은 코퍼스 빈도로 |
| `ts_rewrite` | – | – | 질의 | 표현마다 규칙 · 한 방향으로 |
| 사전 체인 (simple · synonym) | 필요 | 필요 | 문서 · 질의 | 형태소 단위로 적는다 |
| `dict_xsyn` | 필요 | 필요 | 문서 | 질의 쪽에 쓰면 AND |

In [ ]:
# 셀 10 · 정리
T10 = pd.concat([R5.assign(조건=lambda d: "질의 불용어 " + d["불용어 수"].astype(str)),
                 pd.DataFrame(rows9).assign(조건="korean_lab 문서·질의")])[["방법", "조건", "Hit@10", "MRR"]]
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
for a, meth in zip(axes, ["ts_rank (OR)", "BM25"]):
    s = T10[T10["방법"] == meth]
    y = np.arange(len(s))
    a.barh(y + 0.2, s["Hit@10"], 0.4, color="#3D484A", label="Hit@10"); a.barh(y - 0.2, s["MRR"], 0.4, color="#FF7043", label="MRR")
    for i, (h, m) in enumerate(s[["Hit@10", "MRR"]].values):
        a.text(h, i + 0.2, f" {h:.3f}", va="center", fontsize=8); a.text(m, i - 0.2, f" {m:.3f}", va="center", fontsize=8)
    a.set_yticks(y, s["조건"]); a.invert_yaxis(); a.set_xlim(0, 1.1); a.set_title(meth)
axes[0].legend(loc="upper center", bbox_to_anchor=(1.05, -0.08), ncol=2)
plt.tight_layout(); plt.show()
T10.pivot(index="조건", columns="방법", values=["Hit@10", "MRR"]).reindex(T10["조건"].unique())